In [1]:
#efficient net b4
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
# --- Changed imports for EfficientNet ---
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights # Import EfficientNet B4
from torch.utils.data import DataLoader
from tqdm import tqdm
# Import for additional metrics (iqqqq

    print(f"Val Loss: {val_loss/len(val_loader):.4f}, Accuracy: {val_acc:.4f}")
    print(f"Val Precision: {val_precision:.4f}, Val Recall: {val_recall:.4f}, Val F1-Score: {val_f1:.4f}, Val ROC AUC: {val_roc_auc:.4f}")


# Save model
# --- Updated saved model filename ---
torch.save(model.state_dict(), "eye_attention_model_efficientnet_b4_no_aug.pth")
print("Training complete. Model saved as eye_attention_model_efficientnet_b4_no_aug.pth")

IndentationError: unexpected indent (3489102213.py, line 13)

In [ ]:
import cv2
import torch
import torch.nn as nn
from torchvision import transforms
# --- Using EfficientNet B4 as per the provided training code ---
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from PIL import Image # For applying torchvision transforms to OpenCV images
import numpy as np

# --- 1. Parameters (MUST match training parameters) ---
IMG_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# --- Updated MODEL_PATH to match the provided training code's save name ---
MODEL_PATH = "eye_attention_model_efficientnet_b4_no_aug.pth" # Path to your saved model weights

# --- 2. Define the Model Architecture (MUST match training architecture) ---
# Load pretrained EfficientNet B4 Model
weights = EfficientNet_B4_Weights.DEFAULT
model = efficientnet_b4(weights=weights)

# Freeze feature extractor layers (not strictly necessary for inference, but mirrors training setup)
for param in model.features.parameters():
    param.requires_grad = False

# Modify classifier for binary classification (MUST match the one used during training)
num_ftrs = model.classifier[-1].in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 4096),
    nn.ReLU(True),
    nn.Dropout(0.5), # Assuming these dropout rates were used in training
    nn.Linear(4096, 4096),
    nn.ReLU(True),
    nn.Dropout(0.5), # Assuming these dropout rates were used in training
    nn.Linear(4096, 1),
    nn.Sigmoid()
)

# Load the trained weights
try:
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval() # Set model to evaluation mode (IMPORTANT for BatchNorm and Dropout layers)
    print(f"Model loaded successfully from {MODEL_PATH}")
except FileNotFoundError:
    print(f"Error: Model file not found at {MODEL_PATH}. Please ensure the path is correct and the model was saved.")
    exit()
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

# --- 3. Image Transformations for Inference ---
# These should be EXACTLY the same as your 'val' transforms from training
# Since your provided training code used NO augmentation and identical train/val transforms (after the transform overwrite),
# we use the simple Resize, ToTensor, Normalize.
inference_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 4. Prediction Function ---
def predict_attention(image_frame):
    # Convert OpenCV BGR image to PIL RGB image
    # OpenCV reads images as BGR, but torchvision expects RGB
    image_rgb = cv2.cvtColor(image_frame, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(image_rgb)

    # Apply transformations
    input_tensor = inference_transforms(pil_image)
    input_batch = input_tensor.unsqueeze(0) # Add a batch dimension (B, C, H, W)

    # Move to device (CPU or GPU)
    input_batch = input_batch.to(DEVICE)

    with torch.no_grad(): # No need to calculate gradients during inference
        output = model(input_batch)
        prediction_prob = output.item() # Get the single probability value

    # Classify based on a threshold (e.g., 0.5)
    # Assuming 'Attentive' is class 1 (positive) and 'Non-Attentive' is class 0 (negative)
    threshold = 0.5
    if prediction_prob >= threshold:
        prediction_label = "Attentive"
        color = (0, 255, 0) # Green for Attentive (BGR)
    else:
        prediction_label = "Non-Attentive"
        color = (0, 0, 255) # Red for Non-Attentive (BGR)

    return prediction_label, prediction_prob, color

# --- 5. Main Webcam Loop ---
def run_webcam_prediction():
    cap = cv2.VideoCapture(0) # 0 is typically the default webcam

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    print("Webcam opened successfully. Press 'q' to quit.")

    while True:
        ret, frame = cap.read() # Read a frame from the webcam

        if not ret:
            print("Failed to grab frame, exiting...")
            break

        # Get prediction
        label, prob, text_color = predict_attention(frame)

        # Display the prediction on the frame
        display_text = f"{label} ({prob:.2f})"
        cv2.putText(frame, display_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, text_color, 2, cv2.LINE_AA)

        # Display the frame
        cv2.imshow('Webcam Attention Prediction', frame)

        # Exit if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release resources
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam session ended.")

if _name_ == '_main_':
    run_webcam_prediction()